# Cross-Model Answer Consistency: Generation + Verification

## Setup

In [1]:
import json
import random
import re
import sys
import time
import threading
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from tqdm import tqdm
import pandas as pd

sys.path.insert(0, '..')
from utils.api_client import create_openrouter_client
from utils.data_io import load_dataset, save_dataset

# Initialize OpenRouter client (with 3-minute per-request timeout)
client = create_openrouter_client()
client = OpenAI(
    base_url=client.base_url,
    api_key=client.api_key,
    timeout=180,
)

# Per-model token tracking
token_counts: Dict[str, int] = {}
token_lock = threading.Lock()


def track_tokens(model: str, usage) -> None:
    """Thread-safe token tracking per model."""
    if usage is None:
        return
    with token_lock:
        if model not in token_counts:
            token_counts[model] = 0
        token_counts[model] += (usage.prompt_tokens or 0) + (usage.completion_tokens or 0)


print("Setup complete.")

Setup complete.


## Configuration

In [2]:
@dataclass
class CrossModelConfig:
    """Configuration for cross-model generation and verification."""
    # Input / Output
    seed_file: str = "../data/seed_dataset.json"
    output_file: str = "../data/cross_model_verified.jsonl"
    rejected_file: str = "../data/cross_model_rejected.jsonl"
    finetuning_file: str = "../data/cross_model_finetuning.jsonl"

    # Models
    generator_model: str = "anthropic/claude-sonnet-4.5"   # generates Q/R/A
    verifier_model: str = "anthropic/claude-opus-4"                   # solves Q independently
    comparison_model: str = "google/gemini-2.5-flash"       # LLM fallback comparator

    # Generation
    target_count: int = 200
    num_seed_examples: int = 3
    questions_per_batch: int = 2
    generation_temperature: float = 0.8
    verification_temperature: float = 0.1
    max_tokens: int = 25000

    # Parallelism & comparison
    max_workers: int = 5
    numerical_tolerance: float = 0.05   # 5% tolerance for numerical matching

    # Quality thresholds
    min_question_length: int = 30
    min_reasoning_length: int = 50
    min_answer_length: int = 1


config = CrossModelConfig()

print("Configuration:")
print(f"  Seed file:          {config.seed_file}")
print(f"  Generator model:    {config.generator_model}")
print(f"  Verifier model:     {config.verifier_model}")
print(f"  Comparison model:   {config.comparison_model}")
print(f"  Target count:       {config.target_count}")
print(f"  Questions/batch:    {config.questions_per_batch}")
print(f"  Numerical tol.:     {config.numerical_tolerance*100:.0f}%")

Configuration:
  Seed file:          ../data/seed_dataset.json
  Generator model:    anthropic/claude-sonnet-4.5
  Verifier model:     anthropic/claude-opus-4
  Comparison model:   google/gemini-2.5-flash
  Target count:       200
  Questions/batch:    2
  Numerical tol.:     5%


## Load Seed Dataset

In [3]:
seed_dataset = load_dataset(config.seed_file)
print(f"Loaded {len(seed_dataset)} seed examples")

if seed_dataset:
    sample = seed_dataset[0]
    print(f"\nSample seed:")
    print(f"  Q: {sample['question'][:120]}...")
    print(f"  R: {sample['reasoning'][:120]}...")
    print(f"  A: {sample['answer'][:80]}")

Loaded 56 seed examples

Sample seed:
  Q: What is the minimum sample size that will allow us to verify a 500,000-hour MTTF with 85% confidence, given that the tes...
  R: # Reliability Demonstration Test: Minimum Sample Size Calculation

## Problem Statement
We need to determine the minimum...
  A: 380.


## Prompts

In [4]:
# --- Generation prompt (reused from synthetic_data_generation.ipynb) ---

GENERATION_PROMPT = """You are an expert in reliability engineering, creating practice problems for students.

I will show you {num_examples} example problems. Your task is to:
1. Understand the STYLE, DIFFICULTY, and TOPICS of these examples
2. Create {num_to_generate} NEW, ORIGINAL problems that are SIMILAR in style and difficulty
3. Make questions that are EDUCATIONAL and CHALLENGING
4. Provide COMPLETE solutions with step-by-step reasoning

IMPORTANT:
- Create DIFFERENT questions (don't just change numbers in the examples)
- Use similar concepts but NEW scenarios
- Include all necessary data in the question (self-contained)
- Show clear reasoning steps
- Provide specific numerical answers when appropriate

# EXAMPLE PROBLEMS:

{examples}

# YOUR TASK:

Generate {num_to_generate} NEW problems inspired by these examples. For each problem, provide:

QUESTION: [The complete problem statement with all necessary data]

REASONING: [Step-by-step solution showing your work]

ANSWER: [Final answer(s) - be specific]

---

Now generate the {num_to_generate} new problems:"""


def format_examples(examples: List[Dict]) -> str:
    """Format seed examples for the prompt."""
    formatted = []
    for i, ex in enumerate(examples, 1):
        formatted.append(
            f"""## Example {i}:

QUESTION: {ex['question']}

REASONING: {ex['reasoning']}

ANSWER: {ex['answer']}
""")
    return "\n".join(formatted)


def create_generation_prompt(seed_examples: List[Dict], config: CrossModelConfig) -> str:
    """Create the full generation prompt with seed examples."""
    return GENERATION_PROMPT.format(
        num_examples=len(seed_examples),
        num_to_generate=config.questions_per_batch,
        examples=format_examples(seed_examples),
    )


print("Generation prompt defined.")

Generation prompt defined.


In [5]:
# --- Verification solving prompt (modeled on reasoning_processor.py) ---

VERIFICATION_SOLVE_PROMPT = """You are an expert in reliability engineering, statistics, and probability theory.

Solve the following problem step by step. Show your complete reasoning and calculations.

**IMPORTANT**:
- Work through the problem methodically
- Show all formulas used
- Show all calculations
- State your final answer clearly at the end

**Problem:**
{question}

**Your Solution:**"""

print("Verification prompt defined.")

Verification prompt defined.


In [6]:
# --- Answer extraction prompt (reused from model_evaluation.ipynb) ---

EXTRACTION_PROMPT = """You are extracting the final answer from a model's response to a reliability engineering question.

The model may have provided reasoning, calculations, or thinking. Your job is to extract ONLY the final answer.

EXTRACTION RULES:
1. Look for phrases like "final answer", "the answer is", "therefore", "result"
2. Extract the numerical value(s), boolean (True/False), or expression
3. For multi-part answers, provide comma-separated values
4. Remove units, explanations, and extra text
5. If multiple numbers are given, extract all of them comma-separated
6. If no clear answer exists, respond with: "UNABLE_TO_EXTRACT"

EXAMPLES:
Model response: "After calculating, the reliability is 0.95 and the MTTF is 1000 hours."
Your extraction: "0.95, 1000"

Model response: "The system will fail, so the answer is False."
Your extraction: "False"

Model response: "Therefore, the final answer is approximately 0.8413."
Your extraction: "0.8413"

ORIGINAL QUESTION:
{question}

MODEL'S RESPONSE:
{model_response}

Extract ONLY the final answer (no explanation):"""

print("Extraction prompt defined.")

Extraction prompt defined.


In [7]:
# --- LLM comparison prompt (modeled on reasoning_processor.py verification) ---

COMPARISON_PROMPT = """Compare the following two answers to determine if they are essentially the same
(allowing for minor differences in notation, rounding, or phrasing).

**Original Question:**
{question}

**Answer A (Generator):**
{answer_a}

**Answer B (Verifier):**
{answer_b}

**Instructions:**
1. Extract the core numerical answer or conclusion from each
2. Compare them
3. Consider answers as matching if:
   - Numerical values are within 5% of each other
   - The core conclusion/concept is the same
   - Minor notation differences are acceptable

**Response Format:**
Respond with ONLY one of these two words:
- "MATCH" if the answers are essentially the same
- "MISMATCH" if the answers are different

Your response:"""

print("Comparison prompt defined.")

Comparison prompt defined.


In [8]:
# --- Hybrid comparison logic ---

def compare_numerical(answer_a: str, answer_b: str, tolerance: float = 0.05) -> Optional[bool]:
    """Try regex-based numerical comparison.

    Returns True (match), False (mismatch), or None (inconclusive / non-numeric).
    """
    nums_a = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', answer_a)
    nums_b = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', answer_b)

    if not nums_a or not nums_b:
        return None  # non-numeric -- need LLM fallback

    try:
        floats_a = [float(x) for x in nums_a]
        floats_b = [float(x) for x in nums_b]
    except ValueError:
        return None

    if len(floats_a) != len(floats_b):
        return None  # different number of values -- inconclusive

    matches = [
        abs(a - b) / max(abs(b), 1e-10) < tolerance
        for a, b in zip(floats_a, floats_b)
    ]
    return all(matches)


def compare_with_llm(
    question: str, answer_a: str, answer_b: str, config: CrossModelConfig
) -> bool:
    """Fallback: use Gemini Flash to compare two answers."""
    prompt = COMPARISON_PROMPT.format(
        question=question, answer_a=answer_a, answer_b=answer_b
    )
    try:
        response = client.chat.completions.create(
            model=config.comparison_model,
            max_tokens=50,
            temperature=0.0,
            timeout=180,
            messages=[{"role": "user", "content": prompt}],
        )
        track_tokens(config.comparison_model, response.usage)
        content = extract_content_from_message(response.choices[0].message)
        if not content:
            print("  [!] LLM comparison returned no content", flush=True)
            return False
        result = content.strip().upper()
        return "MATCH" in result and "MISMATCH" not in result
    except Exception as e:
        print(f"LLM comparison error: {e}", flush=True)
        return False


def compare_answers_hybrid(
    question: str, answer_a: str, answer_b: str, config: CrossModelConfig
) -> Tuple[bool, str]:
    """Orchestrate hybrid comparison: numerical first, LLM fallback.

    Returns (is_match, method_used).
    """
    numerical_result = compare_numerical(answer_a, answer_b, config.numerical_tolerance)
    if numerical_result is not None:
        return numerical_result, "numerical"

    llm_result = compare_with_llm(question, answer_a, answer_b, config)
    return llm_result, "llm"


print("Hybrid comparison logic defined.")

Hybrid comparison logic defined.


In [9]:
# --- Helper functions ---

def extract_content_from_message(message) -> Optional[str]:
    """Extract usable text content from a chat completion message.

    Reasoning models (e.g. GPT-5 on OpenRouter) may consume the entire
    max_tokens budget on reasoning tokens, leaving message.content as None.
    This helper tries several fallback fields to recover the text.
    """
    # Standard path
    if message.content is not None:
        return message.content

    # Model refused to answer
    if getattr(message, 'refusal', None) is not None:
        print(f"  [!] model refused: {message.refusal}", flush=True)
        return None

    # OpenRouter reasoning model fallbacks (stored in model_extra)
    extras = getattr(message, 'model_extra', None) or {}
    for field in ('reasoning', 'reasoning_content', 'reasoning_details'):
        value = extras.get(field)
        if value and isinstance(value, str) and len(value.strip()) > 0:
            print(f"  [!] content was None; recovered from '{field}' field ({len(value)} chars)", flush=True)
            return value.strip()
        # reasoning_details can be a list of dicts
        if value and isinstance(value, list):
            texts = [v.get('content', '') for v in value if isinstance(v, dict)]
            joined = "\n".join(t for t in texts if t)
            if joined.strip():
                print(f"  [!] content was None; recovered from '{field}' field ({len(joined)} chars)", flush=True)
                return joined.strip()

    # Nothing found -- log available extra keys for debugging
    if extras:
        print(f"  [!] WARNING: content=None and no known reasoning field found. Extra keys: {list(extras.keys())}", flush=True)

    return None


def call_model(
    model: str,
    prompt: str,
    temperature: float = 0.1,
    max_tokens: int = 4096,
    reasoning_effort: Optional[str] = None,
) -> str:
    """Call an OpenRouter model and return the response text.

    Retries up to 3 times with exponential backoff on failure.
    If the model returns content=None (reasoning budget exhaustion),
    tries to recover from reasoning fields before retrying.
    """
    max_retries = 3
    retry_delay = 2.0

    extra_body = {}
    if reasoning_effort is not None:
        extra_body["reasoning"] = {"effort": reasoning_effort}

    for attempt in range(max_retries):
        try:
            kwargs = dict(
                model=model,
                max_tokens=max_tokens,
                temperature=temperature,
                timeout=180,
                messages=[{"role": "user", "content": prompt}],
            )
            if extra_body:
                kwargs["extra_body"] = extra_body

            response = client.chat.completions.create(**kwargs)
            track_tokens(model, response.usage)

            content = extract_content_from_message(response.choices[0].message)

            if content is not None and content.strip():
                print(f"  [{model.split('/')[-1]}] OK", flush=True)
                return content

            # content is None / empty even after recovery -- retry
            wait_time = retry_delay * (2 ** attempt)
            if attempt < max_retries - 1:
                print(f"  [{model.split('/')[-1]}] content empty (attempt {attempt+1}) -- retrying in {wait_time:.0f}s", flush=True)
                time.sleep(wait_time)
            else:
                print(f"  [{model.split('/')[-1]}] WARNING: content empty after {max_retries} attempts", flush=True)
                return ""

        except Exception as e:
            wait_time = retry_delay * (2 ** attempt)
            if attempt < max_retries - 1:
                print(f"  [{model.split('/')[-1]}] attempt {attempt+1} failed: {e} -- retrying in {wait_time:.0f}s", flush=True)
                time.sleep(wait_time)
            else:
                print(f"  [{model.split('/')[-1]}] FAILED after {max_retries} attempts: {e}", flush=True)
                return ""

    return ""


def extract_answer_from_response(question: str, model_response: str, config: CrossModelConfig) -> str:
    """Use Gemini Flash to extract the final answer from a verbose response."""
    prompt = EXTRACTION_PROMPT.format(question=question, model_response=model_response)
    try:
        response = client.chat.completions.create(
            model=config.comparison_model,
            max_tokens=200,
            temperature=0.1,
            timeout=180,
            messages=[{"role": "user", "content": prompt}],
        )
        track_tokens(config.comparison_model, response.usage)
        content = extract_content_from_message(response.choices[0].message)
        return content.strip() if content else "EXTRACTION_ERROR"
    except Exception as e:
        print(f"Extraction error: {e}", flush=True)
        return "EXTRACTION_ERROR"


def parse_generated_output(output: str) -> List[Dict]:
    """Parse the generator's output to extract Q/R/A triplets."""
    results = []
    sections = [s.strip() for s in output.split('QUESTION:') if s.strip()]

    for section in sections:
        reasoning_match = re.search(
            r'REASONING:\s*(.*?)(?=ANSWER:|$)', section, re.IGNORECASE | re.DOTALL
        )
        answer_match = re.search(
            r'ANSWER:\s*(.*?)(?=QUESTION:|---+|$)', section, re.IGNORECASE | re.DOTALL
        )
        question_text = re.split(
            r'(REASONING|ANSWER):', section, flags=re.IGNORECASE, maxsplit=1
        )[0].strip()

        reasoning_text = reasoning_match.group(1).strip() if reasoning_match else ""
        answer_text = answer_match.group(1).strip() if answer_match else ""

        if question_text and reasoning_text and answer_text:
            results.append({
                'question': question_text,
                'reasoning': reasoning_text,
                'answer': answer_text,
            })

    return results


def validate_generated(item: Dict, config: CrossModelConfig) -> Tuple[bool, str]:
    """Validate a generated Q/R/A triplet."""
    if len(item['question']) < config.min_question_length:
        return False, "Question too short"
    if len(item['reasoning']) < config.min_reasoning_length:
        return False, "Reasoning too short"
    if len(item['answer']) < config.min_answer_length:
        return False, "Answer too short"

    invalid_patterns = ['[insert', 'tbd', 'to be determined', 'xxx', '???']
    text = (item['question'] + item['reasoning'] + item['answer']).lower()
    for pattern in invalid_patterns:
        if pattern in text:
            return False, f"Contains placeholder: {pattern}"

    return True, ""


print("Helper functions defined.")

Helper functions defined.


In [10]:
# --- Verify a single question ---

def verify_single_question(item: Dict, config: CrossModelConfig) -> Dict:
    """Send Q to the verifier model, extract both answers, compare.

    Returns the item annotated with verification metadata.
    """
    question = item['question']
    generator_answer = item['answer']

    # 1. Have the verifier solve the question independently
    verifier_prompt = VERIFICATION_SOLVE_PROMPT.format(question=question)
    verifier_response = call_model(
        config.verifier_model, verifier_prompt,
        temperature=config.verification_temperature, max_tokens=16384,
        reasoning_effort="low",
    )

    if not verifier_response:
        item['verified'] = False
        item['verification_method'] = 'error'
        item['verifier_answer'] = ''
        item['reject_reason'] = 'Verifier API call failed'
        return item

    # 2. Extract final answer from both responses
    extracted_generator = extract_answer_from_response(question, item['reasoning'] + "\n" + generator_answer, config)
    extracted_verifier = extract_answer_from_response(question, verifier_response, config)

    # 3. Compare answers (hybrid: numerical first, LLM fallback)
    is_match, method = compare_answers_hybrid(
        question, extracted_generator, extracted_verifier, config
    )

    # 4. Annotate
    item['verified'] = is_match
    item['verification_method'] = method
    item['extracted_generator_answer'] = extracted_generator
    item['extracted_verifier_answer'] = extracted_verifier
    item['verifier_response'] = verifier_response
    if not is_match:
        item['reject_reason'] = f"Answer mismatch ({method}): generator='{extracted_generator}' vs verifier='{extracted_verifier}'"

    return item

In [ ]:
# --- Main pipeline ---

def run_pipeline(seed_dataset: List[Dict], config: CrossModelConfig):
    """Generate, verify, and save until target_count accepted samples are reached.

    Resumable: loads existing accepted/rejected files on start.
    """
    # Resume support: load existing progress
    accepted: List[Dict] = []
    rejected: List[Dict] = []

    if Path(config.output_file).exists():
        accepted = load_dataset(config.output_file)
        print(f"Resumed: {len(accepted)} accepted samples from previous run.", flush=True)
    if Path(config.rejected_file).exists():
        rejected = load_dataset(config.rejected_file)
        print(f"Resumed: {len(rejected)} rejected samples from previous run.", flush=True)

    batch_num = 0
    comparison_methods = {'numerical': 0, 'llm': 0, 'error': 0}

    print(f"\n{'='*80}", flush=True)
    print(f"CROSS-MODEL GENERATION PIPELINE", flush=True)
    print(f"{'='*80}", flush=True)
    print(f"Target: {config.target_count} verified samples", flush=True)
    print(f"Starting from: {len(accepted)} accepted, {len(rejected)} rejected", flush=True)
    print(flush=True)
    sys.stdout.flush()

    start_time = time.time()
    pbar = tqdm(total=config.target_count, initial=len(accepted), desc="Accepted")

    while len(accepted) < config.target_count:
        batch_num += 1

        # Safety valve
        total_processed = len(accepted) + len(rejected)
        if batch_num > 20 and total_processed > 0:
            acceptance_rate = len(accepted) / total_processed
            if acceptance_rate < 0.10:
                print(f"\nWARNING: Acceptance rate dropped to {acceptance_rate:.1%} after {batch_num} batches. Stopping.", flush=True)
                break

        # 1. Sample seeds and generate
        print(f"Batch {batch_num}: generating...", flush=True)
        seeds = random.sample(seed_dataset, min(config.num_seed_examples, len(seed_dataset)))
        gen_prompt = create_generation_prompt(seeds, config)
        raw_output = call_model(
            config.generator_model, gen_prompt,
            temperature=config.generation_temperature, max_tokens=config.max_tokens,
        )

        if not raw_output:
            print(f"Batch {batch_num}: generation failed, skipping.", flush=True)
            continue

        # 2. Parse and validate
        parsed = parse_generated_output(raw_output)
        valid_items = []
        for item in parsed:
            ok, reason = validate_generated(item, config)
            if ok:
                item['source'] = 'cross_model_verified'
                item['batch'] = batch_num
                valid_items.append(item)

        if not valid_items:
            continue

        # 3. Verify in parallel
        print(f"Batch {batch_num}: verifying {len(valid_items)} questions...", flush=True)
        with ThreadPoolExecutor(max_workers=config.max_workers) as executor:
            futures = {
                executor.submit(verify_single_question, item, config): item
                for item in valid_items
            }
            for future in as_completed(futures):
                try:
                    result = future.result()
                except Exception as e:
                    print(f"Verification error: {e}", flush=True)
                    continue

                method = result.get('verification_method', 'error')
                comparison_methods[method] = comparison_methods.get(method, 0) + 1

                if result.get('verified'):
                    accepted.append(result)
                    pbar.update(1)
                else:
                    rejected.append(result)

        print(f"Batch {batch_num}: {len(accepted)} accepted, {len(rejected)} rejected so far", flush=True)

        # 4. Save incrementally
        save_dataset(accepted, config.output_file)
        save_dataset(rejected, config.rejected_file)

    pbar.close()
    elapsed = time.time() - start_time

    print(f"\n{'='*80}", flush=True)
    print(f"PIPELINE COMPLETE", flush=True)
    print(f"{'='*80}", flush=True)
    print(f"Accepted:  {len(accepted)}", flush=True)
    print(f"Rejected:  {len(rejected)}", flush=True)
    print(f"Batches:   {batch_num}", flush=True)
    print(f"Time:      {elapsed/60:.1f} minutes", flush=True)
    print(f"Comparison methods: {comparison_methods}", flush=True)

    return accepted, rejected, comparison_methods


# RUN
accepted, rejected, comparison_methods = run_pipeline(seed_dataset, config)

Resumed: 0 accepted samples from previous run.
Resumed: 4 rejected samples from previous run.

CROSS-MODEL GENERATION PIPELINE
Target: 200 verified samples
Starting from: 0 accepted, 4 rejected



Accepted:   0%|          | 0/200 [00:00<?, ?it/s]

Batch 1: generating...
  [claude-sonnet-4.5] OK
Batch 1: verifying 2 questions...
  [claude-opus-4] OK


Accepted:   0%|          | 1/200 [01:57<6:28:28, 117.13s/it]

  [claude-opus-4] OK


...

  [claude-opus-4] OK


Accepted:  14%|█▍        | 29/200 [52:40<3:23:23, 71.36s/it]

Batch 25: 29 accepted, 25 rejected so far
Batch 26: generating...
  [claude-sonnet-4.5] OK
Batch 26: verifying 2 questions...
  [claude-opus-4] OK


Accepted:  15%|█▌        | 30/200 [54:03<3:32:08, 74.87s/it]

  [claude-opus-4] OK
Batch 26: 30 accepted, 26 rejected so far
Batch 27: generating...
  [claude-sonnet-4.5] OK
Batch 27: verifying 2 questions...
  [claude-opus-4] OK
  [claude-opus-4] OK
Batch 27: 30 accepted, 28 rejected so far
Batch 28: generating...
  [claude-sonnet-4.5] OK
Batch 28: verifying 2 questions...


In [ ]:
# --- Save clean output + fine-tuning format ---

def convert_to_finetuning_format(data: List[Dict]) -> List[Dict]:
    """Convert to fine-tuning messages format."""
    finetuning_data = []
    for item in data:
        assistant_content = f"""Let me solve this step by step.

{item['reasoning']}

Therefore, the answer is: {item['answer']}"""
        finetuning_data.append({
            "messages": [
                {"role": "user", "content": item['question']},
                {"role": "assistant", "content": assistant_content},
            ]
        })
    return finetuning_data


# Save verified dataset (already saved incrementally, but save final clean copy)
save_dataset(accepted, config.output_file)
print(f"Saved {len(accepted)} verified entries to: {config.output_file}")

# Save fine-tuning format
finetuning_data = convert_to_finetuning_format(accepted)
save_dataset(finetuning_data, config.finetuning_file)
print(f"Saved {len(finetuning_data)} fine-tuning entries to: {config.finetuning_file}")

In [ ]:
# --- Statistics ---

total_processed = len(accepted) + len(rejected)
acceptance_rate = len(accepted) / total_processed * 100 if total_processed else 0

print("\n" + "="*80)
print("GENERATION STATISTICS")
print("="*80)

print(f"\nAccepted:        {len(accepted)}")
print(f"Rejected:        {len(rejected)}")
print(f"Acceptance rate: {acceptance_rate:.1f}%")

print(f"\nComparison method breakdown:")
for method, count in comparison_methods.items():
    pct = count / total_processed * 100 if total_processed else 0
    print(f"  {method:12s}: {count:4d} ({pct:.1f}%)")

# Cost estimates (check OpenRouter for exact rates)
COST_PER_1M_TOKENS = {
    'anthropic/claude-sonnet-4.5': 3.00,
    'openai/gpt-5': 2.00,
    'google/gemini-2.5-flash': 0.075,
}

print(f"\nToken usage & estimated cost:")
total_cost = 0
for model, tokens in token_counts.items():
    rate = COST_PER_1M_TOKENS.get(model, 0.50)
    cost = (tokens / 1_000_000) * rate
    total_cost += cost
    print(f"  {model}: {tokens:,} tokens (~${cost:.4f})")
print(f"  TOTAL: ~${total_cost:.4f}")
print("  Note: Check OpenRouter for exact pricing.")

NameError: name 'accepted' is not defined

In [ ]:
# --- Show examples of rejected pairs ---

print("\n" + "="*80)
print("SAMPLE REJECTED PAIRS")
print("="*80)

for i, item in enumerate(rejected[:5], 1):
    print(f"\n--- Rejected {i} ---")
    print(f"Question:  {item['question'][:150]}...")
    print(f"Generator: {item.get('extracted_generator_answer', 'N/A')}")
    print(f"Verifier:  {item.get('extracted_verifier_answer', 'N/A')}")
    print(f"Method:    {item.get('verification_method', 'N/A')}")
    print(f"Reason:    {item.get('reject_reason', 'N/A')}")

if not rejected:
    print("\nNo rejected samples.")


SAMPLE REJECTED PAIRS


NameError: name 'rejected' is not defined